# 01 — Model Comparison (STT / Translation / TTS)

**Purpose:** empirically decide which free-tier STT, Translation, and TTS options to commit to
for the Nepali ↔ English MVP, instead of guessing.

This notebook is meant to be run once (or whenever a new candidate model is worth checking),
and the conclusion should be written back into the README / NOTES.md.

**What we're comparing:**
- STT: `faster-whisper` model sizes (`tiny`, `base`, `small`) on a Nepali sample
- Translation: `deep-translator` (Google) output quality on a few tricky sentences
- TTS: `gTTS` Nepali voice quality / naturalness (listen and judge manually)

> Add a real Nepali audio sample at `../assets/sample_ne.wav` before running the STT cells.


# Setup


In [1]:
import sys
sys.path.append("..")  # so we can import from src/

import time
from src.translator import translate_ne_to_en, translate_en_to_ne
from src.text_to_speech import synthesize_speech
from IPython.display import Audio, display


## 1. STT model size comparison

Compares transcription speed and output across Whisper model sizes.
Swap in your own Nepali audio clip at `SAMPLE_AUDIO` below.


In [9]:
SAMPLE_AUDIO = "../src/nepali_speech.mp3"  # replace with a real Nepali clip

from faster_whisper import WhisperModel

results = {}

for size in ["tiny", "base", "small","medium"]:
    print(f"--- Loading model: {size} ---")
    model = WhisperModel(size, device="cpu", compute_type="int8")

    start = time.time()
    segments, info = model.transcribe(SAMPLE_AUDIO, language="ne", beam_size=5)
    text = " ".join(s.text.strip() for s in segments)
    elapsed = time.time() - start

    results[size] = {"text": text, "seconds": round(elapsed, 2)}
    print(f"[{size}] ({elapsed:.2f}s): {text}\n")

results


--- Loading model: tiny ---
[tiny] (0.49s): Namaste, mirror Nam Ayus Ho, TV Goha Bata Ho.

--- Loading model: base ---
[base] (0.85s): Namaste, mereu nam aayu suho, timi guha bata hoo.

--- Loading model: small ---
[small] (2.81s): नमस्ते मेरो नाम आयूस हो तिमि को हा बात हो

--- Loading model: medium ---
[medium] (6.78s): नमस्ते, मेरो नाम आयुश हो, तिमी गहा बाता हो



{'tiny': {'text': 'Namaste, mirror Nam Ayus Ho, TV Goha Bata Ho.',
  'seconds': 0.49},
 'base': {'text': 'Namaste, mereu nam aayu suho, timi guha bata hoo.',
  'seconds': 0.85},
 'small': {'text': 'नमस्ते मेरो नाम आयूस हो तिमि को हा बात हो',
  'seconds': 2.81},
 'medium': {'text': 'नमस्ते, मेरो नाम आयुश हो, तिमी गहा बाता हो',
  'seconds': 6.78}}